# Clasificación de Lesiones Cutáneas — SSL Domain-Adaptive + Fine-tuning (v2)

**Proyecto de Grado UNAB · Objetivo 2 — versión corregida y defendible.**

Cambios frente a la v1 (que daba macro-F1 0.873):
1. **El SSL AHORA sí alimenta el modelo final.** En la v1 el backbone SimCLR (ResNet18) se entrenaba y se guardaba, pero el clasificador arrancaba de `resnet50(ImageNet)` y nunca lo cargaba → la contribución era decorativa. Aquí el SSL es sobre **ResNet50** (misma arquitectura) y se **carga** en el clasificador.
2. **Split por `lesion_id`, no por `image_id`.** HAM10000 tiene varias fotos de la misma lesión; partir por imagen filtra la misma lesión a train y val (leakage) e infla el número. Usamos `StratifiedGroupKFold` lesión-disjunto.
3. **SSL de verdad (SimSiam sobre ResNet50)** domain-adaptive sobre ISIC2019 + HAM(train) **sin etiquetas**, partiendo de ImageNet. Método negative-free → robusto a batch chico / 1 GPU (validado en la Práctica 1: SimCLR desde cero pierde con cómputo de red de segmentación entrenada con pseudo-máscaras).
4. **Ablación con vs sin SSL** — la evidencia de la contribución para la sustentación.
5. **KPIs de tesis medidos:** sensibilidad melanoma (≥0.80), AUC-ROC macro (≥0.85), ECE (≤0.10) con temperature scaling, y macro-F1 (≥0.65).
6. **Modo screening:** umbral de melanoma movido para garantizar sensibilidad ≥0.80.

**Requisitos:** GPU T4 x2 (Settings → Accelerator) + Add Input: `Skin Cancer MNIST: HAM10000` (kmader) y `ISIC 2019 JPG 224x224 RESIZED` (nischaydnk).

In [ ]:
import io, os, random, time
from pathlib import Path
import numpy as np, pandas as pd
import torch, torch.nn as nn, torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from PIL import Image, ImageFile
from scipy.special import softmax
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import (f1_score, accuracy_score, classification_report,
                             roc_auc_score, roc_curve, confusion_matrix)

ImageFile.LOAD_TRUNCATED_IMAGES = True  # una imagen truncada en Kaggle NO debe tumbar el run

def load_rgb(path):
    """Carga robusta: si una imagen esta corrupta, avisa UNA vez y devuelve placeholder negro."""
    try:
        return Image.open(path).convert("RGB")
    except Exception as e:
        if not getattr(load_rgb, "_warned", False):
            print(f"[WARN] imagen ilegible {path}: {e} (se usa placeholder; se silencian las proximas)")
            load_rgb._warned = True
        return Image.new("RGB", (224, 224), (0, 0, 0))

def set_seed(seed=42):
    torch.manual_seed(seed); torch.cuda.manual_seed_all(seed)
    np.random.seed(seed); random.seed(seed)
    torch.backends.cudnn.benchmark = True  # velocidad; determinismo pleno no es critico aqui

set_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
N_GPU = torch.cuda.device_count()
USE_DP = N_GPU > 1  # T4 x2 -> DataParallel para ~2x throughput sin cambiar resultados
print("PyTorch:", torch.__version__, "| Device:", device, "| GPUs:", N_GPU, "| DataParallel:", USE_DP)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(0))

MEAN = [0.485, 0.456, 0.406]; STD = [0.229, 0.224, 0.225]

# ---- Presupuesto de computo (ajusta si te quedas sin tiempo de sesion) ----
# SMOKE = corrida de HUMO: subsample + 2 epochs, valida TODO el pipeline en ~3-5 min.
# FLUJO RECOMENDADO: (1) SMOKE=True -> Run All. Si llega al final sin error, el pipeline sirve.
#                    (2) SMOKE=False -> Run All real (~45 min con 2 GPUs).
SMOKE       = False
SSL_EPOCHS  = 30    # SimSiam domain-adaptive
SSL_BATCH   = 128   # batch 128 a 224px entra en T4; SimSiam es robusto a batch chico
SSL_RES     = 224   # MISMA resolucion que el fine-tune -> evita BN drift (arregla el linear probe)
SUP_EPOCHS  = 20    # fine-tune supervisado por cada brazo de la ablacion
SUP_BATCH   = 64
SUP_RES     = 224
RUN_SSL     = True  # False = saltar SSL y correr solo baseline ImageNet

if SMOKE:
    SSL_EPOCHS, SUP_EPOCHS = 2, 2
    print("\n*** MODO SMOKE ACTIVO -> numeros NO validos como resultado, solo valida que el pipeline corre entero. ***")

## 1. Auto-localizar datasets (robusto a cómo Kaggle los monte)

In [ ]:
INPUT = Path("/kaggle/input")

# HAM10000: metadata + imágenes
ham_metas = list(INPUT.glob("**/HAM10000_metadata.csv"))
assert ham_metas, "Falta HAM10000 (kmader). Add Input -> Datasets -> Skin Cancer MNIST: HAM10000"
ham_meta_csv = ham_metas[0]
HAM_DIR = ham_meta_csv.parent
img_paths = {p.stem: p for p in HAM_DIR.glob("**/*.jpg")}
print("HAM metadata:", ham_meta_csv)
print("Imágenes HAM:", len(img_paths))

# MEJORA 3 — más datos NO etiquetados para el SSL (el SSL escala con datos).
# Agarra TODAS las imágenes ISIC_*.jpg de CUALQUIER dataset ISIC agregado como
# Input (ISIC 2019, 2020, archive, ...), no solo el de 2019. Excluye la carpeta
# de HAM (tiene su propio split etiquetado) y cualquier id ya presente en HAM.
isic_imgs = {}
for p in INPUT.glob("**/ISIC_*.jpg"):
    if HAM_DIR in p.parents or p.stem in img_paths:
        continue
    isic_imgs[p.stem] = p
print("Imágenes externas NO etiquetadas para SSL (ISIC*):", len(isic_imgs))
print("  Tip: agregá más datasets ISIC como Input (2020/archive) para reforzar el SSL.")

## 2. Metadata, split lesión-disjunto (anti-leakage) y pesos suavizados

**Clave:** agrupamos por `lesion_id`. Todas las fotos de una misma lesión caen del mismo lado (train **o** val, nunca ambos). `StratifiedGroupKFold` además mantiene el balance de clases a nivel de lesión.

In [ ]:
meta = pd.read_csv(ham_meta_csv)
meta = meta[meta.image_id.isin(img_paths)].reset_index(drop=True)
classes = sorted(meta.dx.unique())
cls2idx = {c: i for i, c in enumerate(classes)}
id2label = {r.image_id: cls2idx[r.dx] for r in meta.itertuples()}
print("Clases:", classes)
print(meta.dx.value_counts(), "\n")

all_ids = meta.image_id.tolist()
y       = np.array([id2label[i] for i in all_ids])
groups  = meta.set_index("image_id").loc[all_ids, "lesion_id"].values

sgkf = StratifiedGroupKFold(n_splits=5, shuffle=True, random_state=42)
tr_idx, va_idx = next(sgkf.split(all_ids, y, groups))
tr_ids = [all_ids[i] for i in tr_idx]
va_ids = [all_ids[i] for i in va_idx]

# Verificacion dura: cero lesiones compartidas entre train y val
tr_les = set(meta.set_index("image_id").loc[tr_ids, "lesion_id"])
va_les = set(meta.set_index("image_id").loc[va_ids, "lesion_id"])
overlap = tr_les & va_les
print(f"Train imgs: {len(tr_ids)} | Val imgs: {len(va_ids)} | Lesiones compartidas: {len(overlap)}")
assert len(overlap) == 0, "LEAKAGE: hay lesiones en train y val a la vez"

# ---- SMOKE: achica el problema para validar el pipeline rapido ----
# Subsample estratificado (>=1 por clase, incluye mel para screening/Grad-CAM). Subconjunto de
# conjuntos lesion-disjuntos sigue siendo lesion-disjunto -> no reintroduce leakage.
if SMOKE:
    def _sub(ids, per_class, seed=42):
        yy = np.array([id2label[i] for i in ids]); rng = np.random.RandomState(seed); keep = []
        for c in np.unique(yy):
            ci = np.where(yy == c)[0]
            keep.extend(rng.choice(ci, min(per_class, len(ci)), replace=False))
        return [ids[i] for i in sorted(keep)]
    tr_ids = _sub(tr_ids, 120)   # <= ~840 imgs
    va_ids = _sub(va_ids, 40)    # <= ~280 imgs
    print(f"[SMOKE] subsample -> train {len(tr_ids)} | val {len(va_ids)}")

# Pesos = raiz del inverso de frecuencia (suaviza; evita sobre-predecir clases raras)
counts = np.bincount([id2label[i] for i in tr_ids], minlength=len(classes))
w = torch.tensor((counts.sum() / np.maximum(counts, 1)) ** 0.5, dtype=torch.float32)
w = (w / w.mean()).to(device)
print("Pesos por clase:", {c: round(float(x), 2) for c, x in zip(classes, w.cpu())})

## 3. SSL domain-adaptive — SimSiam sobre ResNet50 (la contribución)

SimSiam (Chen & He, CVPR 2021): aprendizaje contrastivo **sin negativos** (stop-gradient + predictor). No necesita batch de miles ni cola de memoria → corre bien en 1 GPU T4. Partimos del backbone **ImageNet** y lo **continuamos** sobre imágenes de piel sin etiquetas (ISIC2019 + HAM-train) para adaptarlo al dominio dermatológico.

> Solo usamos imágenes de **lesiones de train** de HAM (más ISIC externo). Nada de val entra al SSL.

In [ ]:
ssl_aug = transforms.Compose([
    transforms.RandomResizedCrop(SSL_RES, scale=(0.2, 1.0)),
    transforms.RandomApply([transforms.ColorJitter(0.4, 0.4, 0.4, 0.1)], p=0.8),
    transforms.RandomGrayscale(p=0.2),
    transforms.RandomApply([transforms.GaussianBlur(9, (0.1, 2.0))], p=0.5),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class TwoViewDataset(Dataset):
    """Devuelve dos vistas aumentadas de la misma imagen (sin etiqueta)."""
    def __init__(self, paths, tf):
        self.paths = list(paths); self.tf = tf
    def __len__(self):
        return len(self.paths)
    def __getitem__(self, idx):
        img = load_rgb(self.paths[idx])
        return self.tf(img), self.tf(img)

class SimSiam(nn.Module):
    def __init__(self, dim=2048, pred_dim=512):
        super().__init__()
        bb = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)  # backbone ImageNet
        self.feat_dim = bb.fc.in_features  # 2048
        bb.fc = nn.Identity()
        self.backbone = bb
        # projector: MLP de 3 capas con BN (la ultima sin affine, como el paper)
        self.projector = nn.Sequential(
            nn.Linear(self.feat_dim, dim, bias=False), nn.BatchNorm1d(dim), nn.ReLU(inplace=True),
            nn.Linear(dim, dim, bias=False),           nn.BatchNorm1d(dim), nn.ReLU(inplace=True),
            nn.Linear(dim, dim, bias=False),           nn.BatchNorm1d(dim, affine=False))
        # predictor: MLP de 2 capas (cuello de botella)
        self.predictor = nn.Sequential(
            nn.Linear(dim, pred_dim, bias=False), nn.BatchNorm1d(pred_dim), nn.ReLU(inplace=True),
            nn.Linear(pred_dim, dim))

    def forward(self, x1, x2):
        # x1, x2: (B, 3, R, R)
        z1 = self.projector(self.backbone(x1))  # -> (B, dim)
        z2 = self.projector(self.backbone(x2))  # -> (B, dim)
        p1 = self.predictor(z1)                 # -> (B, dim)
        p2 = self.predictor(z2)
        return p1, p2, z1.detach(), z2.detach()  # stop-gradient en z

def neg_cos(p, z):
    # cosine_similarity ya normaliza ambos; z viene detached (stop-grad)
    return -F.cosine_similarity(p, z, dim=1).mean()

In [ ]:
DA_BACKBONE = "/kaggle/working/backbone_domain_adaptive_r50.pth"

if RUN_SSL:
    set_seed(42)
    # Pool SSL: ISIC (externo) + solo lesiones de TRAIN de HAM (val queda fuera)
    ssl_paths = list(isic_imgs.values()) + [img_paths[i] for i in tr_ids]
    print(f"Imagenes en el pool SSL (sin etiquetas): {len(ssl_paths)}")
    ssl_dl = DataLoader(TwoViewDataset(ssl_paths, ssl_aug), batch_size=SSL_BATCH,
                        shuffle=True, num_workers=2, pin_memory=True, drop_last=True)

    ssl_model = SimSiam().to(device)
    if USE_DP:
        ssl_model = nn.DataParallel(ssl_model)  # reparte el batch entre las 2 T4
    base_lr = 0.05 * SSL_BATCH / 256
    ssl_opt = torch.optim.SGD(ssl_model.parameters(), lr=base_lr, momentum=0.9, weight_decay=1e-4)
    ssl_sched = torch.optim.lr_scheduler.CosineAnnealingLR(ssl_opt, T_max=SSL_EPOCHS)
    scaler = torch.amp.GradScaler("cuda")

    def _ssl_backbone():
        # con DataParallel el modelo real vive en .module -> guardar SIEMPRE llaves planas
        return (ssl_model.module if USE_DP else ssl_model).backbone

    t0 = time.time()
    for e in range(SSL_EPOCHS):
        ssl_model.train(); tot = 0.0; last_z = None
        for v1, v2 in ssl_dl:
            v1, v2 = v1.to(device, non_blocking=True), v2.to(device, non_blocking=True)
            ssl_opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                p1, p2, z1, z2 = ssl_model(v1, v2)
                loss = 0.5 * neg_cos(p1, z2) + 0.5 * neg_cos(p2, z1)
            scaler.scale(loss).backward(); scaler.step(ssl_opt); scaler.update()
            tot += loss.item(); last_z = z1
        ssl_sched.step()
        # Diagnostico de colapso SimSiam: std de la representacion L2-normalizada por dimension.
        # Sano ~ 1/sqrt(dim) = 0.022 ; si out_std -> ~0 el modelo colapso (features triviales).
        with torch.no_grad():
            zc = F.normalize(last_z.float(), dim=1)
            out_std = zc.std(dim=0).mean().item()
        el = time.time() - t0; eta = el / (e + 1) * (SSL_EPOCHS - e - 1)
        # loss: hacia -1.0 es bueno (ideal ~ -0.9). Si loss~0 Y out_std~0 -> colapso.
        print(f"[SSL] Epoch {e+1}/{SSL_EPOCHS} - loss: {tot/len(ssl_dl):.4f} | "
              f"out_std: {out_std:.4f} (colapso si ~0, sano ~0.022) | "
              f"lr: {ssl_sched.get_last_lr()[0]:.2e} | ETA: {eta/60:.1f} min")
        if (e + 1) % 10 == 0:  # checkpoint parcial por si la sesion de Kaggle se cae a mitad
            torch.save(_ssl_backbone().state_dict(), DA_BACKBONE)

    torch.save(_ssl_backbone().state_dict(), DA_BACKBONE)
    print(f"Backbone domain-adaptive (ResNet50) guardado en {DA_BACKBONE} | "
          f"tiempo SSL: {(time.time()-t0)/60:.1f} min")
    del ssl_model; torch.cuda.empty_cache()
else:
    print("SSL omitido (RUN_SSL=False).")

## 4. Fine-tune supervisado + utilidades de evaluación (KPIs de tesis)

Toda la artillería: 224px, cosine LR, label smoothing, RandomErasing, rotación/flips, pesos suavizados, TTA (hflip), guardado del mejor epoch por macro-F1. La función `build_model` decide el punto de partida: **ImageNet** o **SSL domain-adaptive**.

In [ ]:
train_tf = transforms.Compose([
    transforms.RandomResizedCrop(SUP_RES, scale=(0.6, 1.0)),
    transforms.RandomHorizontalFlip(), transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.05),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
    transforms.RandomErasing(p=0.25),
])
val_tf = transforms.Compose([
    transforms.Resize(256), transforms.CenterCrop(SUP_RES),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

class HAMds(Dataset):
    def __init__(self, ids, tf): self.ids, self.tf = ids, tf
    def __len__(self): return len(self.ids)
    def __getitem__(self, idx):
        i = self.ids[idx]
        # devuelve (imagen, label). label int
        return self.tf(load_rgb(img_paths[i])), id2label[i]

train_dl = DataLoader(HAMds(tr_ids, train_tf), batch_size=SUP_BATCH, shuffle=True,
                      num_workers=4, pin_memory=True, drop_last=True)
val_dl = DataLoader(HAMds(va_ids, val_tf), batch_size=128, num_workers=4, pin_memory=True)

def build_model(init="imagenet", ssl_path=DA_BACKBONE):
    """init='imagenet' (baseline) o 'ssl' (backbone domain-adaptive)."""
    if init == "ssl":
        net = resnet50(weights=None)
        sd = torch.load(ssl_path, map_location="cpu")
        missing, unexpected = net.load_state_dict(sd, strict=False)  # fc.* falta (esperado)
        assert all(k.startswith("fc.") for k in missing), f"Faltan pesos inesperados: {missing}"
    else:
        net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    net.fc = nn.Linear(net.fc.in_features, len(classes))
    return net.to(device)

@torch.no_grad()
def collect_logits(net, dl):
    """Logits promediados con TTA (hflip). Devuelve (N, C) y labels (N,)."""
    net.eval(); LG, GT = [], []
    for imgs, lbls in dl:
        imgs = imgs.to(device)
        with torch.amp.autocast("cuda"):
            logits = (net(imgs) + net(torch.flip(imgs, dims=[3]))) / 2
        LG.append(logits.float().cpu()); GT.append(lbls)
    return torch.cat(LG).numpy(), torch.cat(GT).numpy()

def ece_score(probs, labels, n_bins=15):
    """Expected Calibration Error."""
    conf = probs.max(1); pred = probs.argmax(1)
    acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); ece = 0.0
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0:
            ece += (m.mean()) * abs(acc[m].mean() - conf[m].mean())
    return float(ece)

def metrics_from_logits(logits, labels, temp=1.0):
    probs = softmax(logits / temp, axis=1)
    pred = probs.argmax(1)
    mf1 = f1_score(labels, pred, average="macro")
    acc = accuracy_score(labels, pred)
    try:
        auc = roc_auc_score(labels, probs, multi_class="ovr", average="macro",
                            labels=list(range(len(classes))))
    except Exception:
        auc = float("nan")
    ece = ece_score(probs, labels)
    return dict(macro_f1=mf1, acc=acc, auc=auc, ece=ece), probs, pred

def train_arm(init, epochs=SUP_EPOCHS, tag=""):
    """Entrena un brazo (imagenet|ssl). Devuelve (best_f1, best_state, val_logits, val_labels)."""
    set_seed(42)
    net = build_model(init)
    if USE_DP:
        net = nn.DataParallel(net)  # 2 GPUs; el batch se reparte solo
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    opt = torch.optim.AdamW(net.parameters(), lr=3e-4, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=epochs)
    scaler = torch.amp.GradScaler("cuda")
    best_f1, best_state, best_logits, best_labels = 0.0, None, None, None
    lg, gt = None, None
    t0 = time.time()
    for e in range(epochs):
        net.train()
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = crit(net(imgs), lbls)
            scaler.scale(loss).backward(); scaler.step(opt); scaler.update()
        sched.step()
        lg, gt = collect_logits(net, val_dl)
        m, _, _ = metrics_from_logits(lg, gt)
        if m["macro_f1"] > best_f1:
            best_f1 = m["macro_f1"]
            src = net.module if USE_DP else net   # guardar SIEMPRE llaves planas (sin 'module.')
            best_state = {k: v.detach().cpu().clone() for k, v in src.state_dict().items()}
            best_logits, best_labels = lg, gt
        el = time.time() - t0; eta = el / (e + 1) * (epochs - e - 1)
        print(f"[{tag}] Epoch {e+1}/{epochs} - acc: {m['acc']:.3f} | macro-F1: {m['macro_f1']:.3f} | "
              f"AUC: {m['auc']:.3f} | lr: {sched.get_last_lr()[0]:.1e} | ETA: {eta/60:.1f} min")
    # Fallback: si nunca mejoro (raro), guarda el ultimo estado para no romper export/eval
    if best_state is None:
        src = net.module if USE_DP else net
        best_state = {k: v.detach().cpu().clone() for k, v in src.state_dict().items()}
        best_logits, best_labels = lg, gt
    del net; torch.cuda.empty_cache()
    return best_f1, best_state, best_logits, best_labels

In [ ]:
# ============================================================================
# MEJORA 1+2 — Fine-tune LP-FT + Layer-wise LR Decay (LLRD)
# ----------------------------------------------------------------------------
# Kumar et al. 2022 ("Fine-tuning can distort pretrained features and
# underperform OOD"): el full fine-tune INGENUO distorsiona las features
# pre-entrenadas -> borra la ventaja del SSL (por eso empataba con ImageNet).
# LP-FT la preserva (+1% en-distribucion / +7-10% fuera-de-distribucion):
#   Fase 1 (LP): backbone congelado, se entrena SOLO la cabeza lineal.
#   Fase 2 (FT): se descongela todo, LR chico + layer-wise LR decay.
# Se aplica a AMBOS brazos (comparacion justa). Esta celda REDEFINE train_arm.
# ============================================================================
LP_EPOCHS  = 2 if SMOKE else 10     # linear probe (cabeza sola)
FT_EPOCHS  = 2 if SMOKE else 15     # fine-tune completo con LLRD
LP_LR      = 1e-3                    # LR alto para la cabeza (backbone congelado)
FT_LR      = 1e-4                    # LR base chico para no distorsionar features
LLRD_DECAY = 0.75                    # decay de LR por bloque (fc alto -> conv1 bajo)

def _llrd_groups(model, base_lr, decay):
    """Grupos de parametros con LR decreciente hacia las capas tempranas."""
    order = ["fc", "layer4", "layer3", "layer2", "layer1", "bn1", "conv1"]
    groups, seen = [], set()
    for depth, prefix in enumerate(order):
        ps = [p for n, p in model.named_parameters()
              if n.split(".")[0] == prefix and p.requires_grad and id(p) not in seen]
        for p in ps:
            seen.add(id(p))
        if ps:
            groups.append({"params": ps, "lr": base_lr * (decay ** depth)})
    rest = [p for _, p in model.named_parameters()
            if p.requires_grad and id(p) not in seen]
    if rest:
        groups.append({"params": rest, "lr": base_lr * (decay ** len(order))})
    return groups

def _run_phase(net, core, epochs, opt, sched, crit, tag, best):
    scaler = torch.amp.GradScaler("cuda")
    t0 = time.time()
    for e in range(epochs):
        net.train()
        for imgs, lbls in train_dl:
            imgs, lbls = imgs.to(device), lbls.to(device)
            opt.zero_grad(set_to_none=True)
            with torch.amp.autocast("cuda"):
                loss = crit(net(imgs), lbls)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
        if sched is not None:
            sched.step()
        lg, gt = collect_logits(net, val_dl)
        m, _, _ = metrics_from_logits(lg, gt)
        if m["macro_f1"] > best["f1"]:
            best["f1"] = m["macro_f1"]
            best["state"] = {k: v.detach().cpu().clone() for k, v in core.state_dict().items()}
            best["lg"], best["gt"] = lg, gt
        eta = (time.time() - t0) / (e + 1) * (epochs - e - 1)
        print(f"[{tag}] Epoch {e+1}/{epochs} - macro-F1: {m['macro_f1']:.3f} | "
              f"AUC: {m['auc']:.3f} | best: {best['f1']:.3f} | ETA {eta/60:.1f} min")

def train_arm(init, epochs=None, tag=""):
    """LP-FT (linear probe -> fine-tune con LLRD). Preserva la representación SSL.
    Devuelve (best_f1, best_state, val_logits, val_labels) — misma firma que antes."""
    set_seed(42)
    net = build_model(init)      # resnet50 (backbone init) + cabeza nueva
    core = net                   # modelo pelado (llaves planas para guardar)
    if USE_DP:
        net = nn.DataParallel(net)
    crit = nn.CrossEntropyLoss(weight=w, label_smoothing=0.1)
    best = {"f1": 0.0, "state": None, "lg": None, "gt": None}

    # Fase 1 — LINEAR PROBE: solo la cabeza, backbone congelado
    for p in core.parameters():
        p.requires_grad = False
    for p in core.fc.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(core.fc.parameters(), lr=LP_LR, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=LP_EPOCHS)
    _run_phase(net, core, LP_EPOCHS, opt, sched, crit, f"{tag}-LP", best)

    # Fase 2 — FINE-TUNE completo con layer-wise LR decay
    for p in core.parameters():
        p.requires_grad = True
    opt = torch.optim.AdamW(_llrd_groups(core, FT_LR, LLRD_DECAY), weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, T_max=FT_EPOCHS)
    _run_phase(net, core, FT_EPOCHS, opt, sched, crit, f"{tag}-FT", best)

    if best["state"] is None:   # fallback (no debería pasar)
        best["state"] = {k: v.detach().cpu().clone() for k, v in core.state_dict().items()}
    del net, core
    torch.cuda.empty_cache()
    return best["f1"], best["state"], best["lg"], best["gt"]

print(f"train_arm redefinido -> LP-FT + LLRD (Kumar 2022). "
      f"LP: {LP_EPOCHS} ep | FT: {FT_EPOCHS} ep | LLRD decay: {LLRD_DECAY}")

## 5. Ablación — Baseline ImageNet  vs  SSL domain-adaptive

Mismo split lesión-disjunto, mismos hiperparámetros, misma semilla. La **única** diferencia es el punto de partida del backbone. Esto aísla el efecto de la contribución SSL.

In [ ]:
results = {}

print("===== BRAZO A: Baseline ImageNet =====")
f1_a, state_a, lg_a, gt_a = train_arm("imagenet", tag="IMAGENET")
results["imagenet"] = (f1_a, state_a, lg_a, gt_a)

if RUN_SSL:
    print("\n===== BRAZO B: SSL domain-adaptive =====")
    f1_b, state_b, lg_b, gt_b = train_arm("ssl", tag="SSL")
    results["ssl"] = (f1_b, state_b, lg_b, gt_b)

print("\n\n================ ABLACIÓN (mejor epoch por brazo) ================")
print(f"{'Backbone':<24}{'macro-F1':>10}{'AUC':>8}{'ECE':>8}")
for name in (["imagenet", "ssl"] if RUN_SSL else ["imagenet"]):
    _, _, lg, gt = results[name]
    m, _, _ = metrics_from_logits(lg, gt)
    label = "ImageNet (baseline)" if name == "imagenet" else "SSL domain-adaptive"
    print(f"{label:<24}{m['macro_f1']:>10.3f}{m['auc']:>8.3f}{m['ece']:>8.3f}")
if RUN_SSL:
    delta = results['ssl'][0] - results['imagenet'][0]
    print(f"\nΔ macro-F1 (SSL - baseline): {delta:+.3f}")

# La ablación se reporta COMPLETA arriba (honestidad científica). Pero el MODELO
# ENTREGABLE del proyecto es el SSL domain-adaptive: es el núcleo de la tesis.
# Todo el resto del pipeline (reporte clínico, calibración, screening, fiabilidad,
# Grad-CAM, export y resumen) se ejecuta SOBRE EL MODELO SSL, no sobre el ganador
# numérico de la ablación (que puede ser ImageNet por ruido de ~0.001).
ablation_winner = max(results, key=lambda k: results[k][0])
print(f"\nGanador de la ablación (solo por macro-F1) -> {ablation_winner} "
      f"(macro-F1 {results[ablation_winner][0]:.3f})")

DEPLOY = "ssl" if RUN_SSL else "imagenet"   # <-- modelo entregable = SSL
best_name = DEPLOY
best_f1, best_state, val_logits, val_labels = results[best_name]
print(f"MODELO ENTREGABLE (se evalúa, calibra y exporta) -> {best_name} "
      f"(macro-F1 {best_f1:.3f})")
if RUN_SSL and ablation_winner != "ssl":
    print("Nota: ImageNet ganó por ruido en macro-F1, pero se despliega el SSL "
          "(núcleo del proyecto). El SSL suele ganar en AUC — ver ablación.")

## 6. Reporte clínico completo + calibración (ECE) + screening de melanoma

KPIs de tesis: **macro-F1 ≥ 0.65**, **AUC-ROC macro ≥ 0.85**, **sensibilidad melanoma ≥ 0.80**, **ECE ≤ 0.10**.

- *Temperature scaling*: ajusta un escalar T sobre los logits de val para calibrar las probabilidades (baja el ECE) sin cambiar el argmax → no afecta accuracy/F1.
- *Modo screening*: como una herramienta de tamizaje debe minimizar falsos negativos de melanoma, movemos el umbral de la probabilidad de `mel` hasta garantizar sensibilidad ≥ 0.80, reportando la especificidad resultante.

In [ ]:
# ---- Reporte por clase (recall = sensibilidad por clase) ----
m0, probs0, pred0 = metrics_from_logits(val_logits, val_labels)
print(f"Backbone ganador: {best_name} | macro-F1: {m0['macro_f1']:.3f} | acc: {m0['acc']:.3f} | "
      f"AUC: {m0['auc']:.3f} | ECE(sin calibrar): {m0['ece']:.3f}\n")
print(classification_report(val_labels, pred0, target_names=classes, digits=3, zero_division=0))
print("Matriz de confusión (filas=real, cols=pred):")
print(pd.DataFrame(confusion_matrix(val_labels, pred0), index=classes, columns=classes))

# ---- Temperature scaling (calibración) ----
def fit_temperature(logits, labels):
    lg = torch.tensor(logits); lb = torch.tensor(labels)
    T = torch.nn.Parameter(torch.ones(1))
    opt = torch.optim.LBFGS([T], lr=0.05, max_iter=100)
    def closure():
        opt.zero_grad(); loss = F.cross_entropy(lg / T.clamp(min=1e-2), lb); loss.backward(); return loss
    opt.step(closure)
    return float(T.detach().clamp(min=1e-2))

T = fit_temperature(val_logits, val_labels)
m_cal, probs_cal, _ = metrics_from_logits(val_logits, val_labels, temp=T)
print(f"\nTemperatura óptima T = {T:.3f}")
print(f"ECE  antes: {m0['ece']:.3f}  ->  después: {m_cal['ece']:.3f}")

# ---- Modo screening: umbral de melanoma para sensibilidad >= 0.80 ----
mi = classes.index("mel")
y_mel = (val_labels == mi).astype(int)
fpr, tpr, thr = roc_curve(y_mel, probs_cal[:, mi])
ok = np.where(tpr >= 0.80)[0]
if len(ok):
    j = ok[0]
    print(f"\n[Screening mel] umbral prob>={thr[j]:.3f} -> sensibilidad={tpr[j]:.3f} | "
          f"especificidad={1-fpr[j]:.3f}")
else:
    print("\n[Screening mel] no se alcanza sens>=0.80 en la curva ROC (revisar).")
sens_argmax = (pred0[val_labels == mi] == mi).mean()
print(f"[Argmax mel] sensibilidad con decisión estándar: {sens_argmax:.3f}")

# ---- Tablero de KPIs ----
print("\n================= KPIs DE TESIS =================")
def kpi(name, val, thr_, ge=True):
    ok = val >= thr_ if ge else val <= thr_
    print(f"  {name:<34}{val:>7.3f}  (meta {'>=' if ge else '<='}{thr_})  {'PASA' if ok else 'NO PASA'}")
kpi("macro-F1", m0["macro_f1"], 0.65)
kpi("AUC-ROC macro", m0["auc"], 0.85)
kpi("Sensibilidad melanoma (screening)", tpr[ok[0]] if len(ok) else sens_argmax, 0.80)
kpi("ECE (calibrado)", m_cal["ece"], 0.10, ge=False)

## 6b. Prueba REAL de la contribución SSL — Linear probe + pocas etiquetas

El full fine-tune de la sección 5 **re-escribe** el backbone, así que tapa la diferencia entre partir de ImageNet o de SSL (por eso el ΔF1 salió ≈0). Para aislar la **calidad de la representación** se usa el protocolo estándar de SSL: **linear probe** (backbone congelado, se entrena solo un clasificador lineal sobre las features extraídas).

Además barremos el **régimen de pocas etiquetas** (10/20/50/100%): el SSL rinde más cuando hay menos datos etiquetados. Si ΔF1 > 0 (sobre todo a 10-20%), esa es la evidencia dura de que el SSL domain-adaptive aporta — la figura que sustenta la contribución en la tesis.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def build_backbone(init):
    """ResNet50 con fc=Identity, punto de partida imagenet|ssl. Solo extractor de features."""
    if init == "ssl":
        net = resnet50(weights=None)
        net.load_state_dict(torch.load(DA_BACKBONE, map_location="cpu"), strict=False)
    else:
        net = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)
    net.fc = nn.Identity()
    return net.to(device).eval()

@torch.no_grad()
def extract_features(backbone, ids):
    # features deterministas (val_tf, sin aug) para train y val -> mide la representación
    dl = DataLoader(HAMds(ids, val_tf), batch_size=128, num_workers=4, pin_memory=True)
    FE, YE = [], []
    for imgs, lbls in dl:
        imgs = imgs.to(device)
        with torch.amp.autocast("cuda"):
            f = backbone(imgs)  # -> (B, 2048)
        FE.append(f.float().cpu()); YE.append(lbls)
    return torch.cat(FE).numpy(), torch.cat(YE).numpy()

def strat_subsample(y, frac, seed=42):
    rng = np.random.RandomState(seed); idx = []
    for c in np.unique(y):
        ci = np.where(y == c)[0]
        n = max(1, int(round(len(ci) * frac)))
        idx.extend(rng.choice(ci, n, replace=False))
    return np.array(idx)

FRACS = [0.1, 0.2, 0.5, 1.0]
arms = ["imagenet", "ssl"] if RUN_SSL else ["imagenet"]
probe = {a: {} for a in arms}

for init in arms:
    bb = build_backbone(init)
    Xtr, ytr = extract_features(bb, tr_ids)
    Xva, yva = extract_features(bb, va_ids)
    del bb; torch.cuda.empty_cache()
    sc = StandardScaler().fit(Xtr)
    Xtr_s, Xva_s = sc.transform(Xtr), sc.transform(Xva)
    for frac in FRACS:
        sub = strat_subsample(ytr, frac)
        clf = LogisticRegression(max_iter=2000, class_weight="balanced", C=1.0)
        clf.fit(Xtr_s[sub], ytr[sub])
        prob = clf.predict_proba(Xva_s); pred = prob.argmax(1)
        mf1 = f1_score(yva, pred, average="macro")
        auc = roc_auc_score(yva, prob, multi_class="ovr", average="macro",
                            labels=list(range(len(classes))))
        probe[init][frac] = (mf1, auc)
        print(f"[LP {init:>8}] labels={int(frac*100):>3}% -> macro-F1: {mf1:.3f} | "
              f"AUC: {auc:.3f} | n_train={len(sub)}")

if RUN_SSL:
    print("\n===== LINEAR PROBE: SSL domain-adaptive vs ImageNet (backbone congelado) =====")
    print(f"{'% labels':>9}{'IN F1':>9}{'SSL F1':>9}{'dF1':>8}{'IN AUC':>9}{'SSL AUC':>9}")
    for frac in FRACS:
        inf, ina = probe["imagenet"][frac]
        ssf, ssa = probe["ssl"][frac]
        print(f"{int(frac*100):>8}%{inf:>9.3f}{ssf:>9.3f}{ssf-inf:>+8.3f}{ina:>9.3f}{ssa:>9.3f}")
    print("\ndF1 > 0 (sobre todo a 10-20% labels) => el SSL domain-adaptive aporta representación superior a ImageNet.")

## 6c. Fiabilidad y sesgo — calibración, intervalos de confianza, abstención, validación externa y Grad-CAM

Una herramienta médica de **apoyo** (no da diagnóstico final) se defiende con 5 evidencias:
1. **Reliability diagram** — que cuando dice "80% seguro" acierte ~80%.
2. **Intervalos de confianza (bootstrap)** — reportar `macro-F1 = 0.74 ± IC95%`, no un número pelado.
3. **Abstención** — si no está seguro, responde *"indeterminado → derivar a especialista"*.
4. **Validación externa** — testear en OTRO dataset (ISIC/BCN) que el modelo nunca vio. Es la prueba más fuerte de generalización.
5. **Grad-CAM** — mostrar que el modelo mira LA LESIÓN y no un pelo, una regla o un marcador (HAM10000 tiene esos artefactos → riesgo de *shortcut learning*).

> Sesgo que NO podemos esconder: HAM10000 es piel clara (Fitzpatrick I-III). No podemos afirmar rendimiento en fototipos IV-VI sin datos regionales (ej. test set BOSQUE de Bogotá). Se declara como limitación + trabajo futuro.

In [ ]:
import matplotlib.pyplot as plt

# Reconstruimos el modelo ganador desde best_state (para eval externa y Grad-CAM)
eval_net = resnet50(weights=None)
eval_net.fc = nn.Linear(eval_net.fc.in_features, len(classes))
eval_net.load_state_dict(best_state); eval_net.to(device).eval()

# ============ 6c.1 Reliability diagram (calibracion visual) ============
def reliability_diagram(probs, labels, n_bins=12, title="Calibracion", fname=None):
    conf = probs.max(1); pred = probs.argmax(1); acc = (pred == labels).astype(float)
    bins = np.linspace(0, 1, n_bins + 1); xs, ys = [], []
    for i in range(n_bins):
        m = (conf > bins[i]) & (conf <= bins[i + 1])
        if m.sum() > 0:
            xs.append(conf[m].mean()); ys.append(acc[m].mean())
    plt.figure(figsize=(4, 4))
    plt.plot([0, 1], [0, 1], "--", color="gray", label="perfecto")
    plt.plot(xs, ys, "o-", label="modelo")
    plt.xlabel("Confianza media"); plt.ylabel("Exactitud real"); plt.title(title)
    plt.legend(); plt.grid(alpha=0.3)
    if fname: plt.savefig(fname, dpi=150, bbox_inches="tight")
    plt.show()

reliability_diagram(probs_cal, val_labels, title=f"Calibracion ({best_name}, T={T:.2f})",
                    fname="/kaggle/working/reliability_diagram.png")

# ============ 6c.2 Intervalos de confianza (bootstrap 1000x) ============
def bootstrap_ci(logits, labels, metric_fn, n=1000, seed=42):
    rng = np.random.RandomState(seed); N = len(labels); vals = []
    for _ in range(n):
        idx = rng.randint(0, N, N)
        vals.append(metric_fn(logits[idx], labels[idx]))
    return float(np.mean(vals)), float(np.percentile(vals, 2.5)), float(np.percentile(vals, 97.5))

f1_fn = lambda lg, lb: f1_score(lb, softmax(lg, axis=1).argmax(1), average="macro")
mel_fn = lambda lg, lb: ((softmax(lg, axis=1).argmax(1) == mi)[lb == mi]).mean() if (lb == mi).any() else np.nan
mu_f1, lo_f1, hi_f1 = bootstrap_ci(val_logits, val_labels, f1_fn)
mu_ms, lo_ms, hi_ms = bootstrap_ci(val_logits, val_labels, mel_fn)
print(f"macro-F1        : {mu_f1:.3f}  IC95% [{lo_f1:.3f}, {hi_f1:.3f}]")
print(f"sens mel (argmax): {mu_ms:.3f}  IC95% [{lo_ms:.3f}, {hi_ms:.3f}]")

# ============ 6c.3 Abstencion (selective prediction) ============
print("\n== Abstencion: responde solo si confia; el resto -> especialista ==")
conf = probs_cal.max(1); pred_c = probs_cal.argmax(1); abst = {}
for tau in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    keep = conf >= tau
    cov = float(keep.mean())
    acc = float((pred_c[keep] == val_labels[keep]).mean()) if keep.sum() else float("nan")
    abst[tau] = (cov, acc)
    print(f"  umbral>={tau:.1f} -> cobertura {cov*100:5.1f}% | exactitud en lo respondido {acc*100:5.1f}%")

# ============ 6c.4 Validacion externa (si hay dataset etiquetado adjunto) ============
ISIC2HAM = {"MEL": "mel", "NV": "nv", "BCC": "bcc", "AK": "akiec", "BKL": "bkl", "DF": "df", "VASC": "vasc"}
ext_result = None
gt_csvs = list(INPUT.glob("**/*GroundTruth*.csv"))
try:
    if gt_csvs:
        gt = pd.read_csv(gt_csvs[0]); idcol = gt.columns[0]
        lblcols = [c for c in gt.columns if c.upper() in ISIC2HAM]
        gt = gt.copy()
        gt["labname"] = [lblcols[i].upper() for i in gt[lblcols].values.argmax(1)]
        gt = gt[gt["labname"].isin(ISIC2HAM)]
        gt["y"] = gt["labname"].map(lambda k: cls2idx[ISIC2HAM[k]])
        ext_paths = {p.stem: p for p in INPUT.glob("**/ISIC_*.jpg")}
        rows = gt[gt[idcol].isin(ext_paths)]
        print(f"\n[Validacion externa] {len(rows)} imagenes externas etiquetadas")
        if len(rows) > 200:
            class ExtDS(Dataset):
                def __init__(self, ids, ys): self.ids, self.ys = ids, ys
                def __len__(self): return len(self.ids)
                def __getitem__(self, i):
                    return val_tf(load_rgb(ext_paths[self.ids[i]])), self.ys[i]
            edl = DataLoader(ExtDS(rows[idcol].tolist(), rows["y"].tolist()),
                             batch_size=128, num_workers=4, pin_memory=True)
            elg, egt = collect_logits(eval_net, edl)
            em, _, _ = metrics_from_logits(elg, egt, temp=T)
            ext_result = {"n": int(len(rows)), **{k: float(v) for k, v in em.items()}}
            print(f"[Validacion externa] macro-F1: {em['macro_f1']:.3f} | AUC: {em['auc']:.3f} | acc: {em['acc']:.3f}")
except Exception as e:
    print("[Validacion externa] error:", e)
if ext_result is None:
    print("\n[Validacion externa] OMITIDA -- adjunta ISIC2019 con su *GroundTruth*.csv (o BCN20000 etiquetado)")
    print("  para medir generalizacion cross-dataset. Es la prueba de fiabilidad mas fuerte.")

In [ ]:
# ============ 6c.5 Grad-CAM: el modelo mira la lesion o un artefacto? ============
class GradCAM:
    def __init__(self, model, layer):
        self.model = model; self.a = None; self.g = None
        layer.register_forward_hook(lambda m, i, o: setattr(self, "a", o.detach()))
        layer.register_full_backward_hook(lambda m, gi, go: setattr(self, "g", go[0].detach()))
    def __call__(self, x, cls):
        self.model.zero_grad()
        out = self.model(x)
        out[0, cls].backward()
        w = self.g.mean(dim=(2, 3), keepdim=True)
        cam = (w * self.a).sum(1, keepdim=True).clamp(min=0)
        cam = F.interpolate(cam, size=x.shape[2:], mode="bilinear", align_corners=False)[0, 0]
        cam = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam.cpu().numpy(), float(out.softmax(1)[0, cls])

def denorm(x):
    t = x.detach().cpu().numpy().transpose(1, 2, 0)
    return np.clip(t * np.array(STD) + np.array(MEAN), 0, 1)

cam = GradCAM(eval_net, eval_net.layer4[-1])
mel_val_ids = [i for i in va_ids if id2label[i] == mi][:3]
if not mel_val_ids:
    print("[Grad-CAM] no hay melanomas en val (posible en SMOKE con subsample chico) -> se omite.")
else:
    fig, axes = plt.subplots(len(mel_val_ids), 2, figsize=(6, 3 * len(mel_val_ids)))
    if len(mel_val_ids) == 1:
        axes = axes[None, :]
    for r, iid in enumerate(mel_val_ids):
        x = val_tf(load_rgb(img_paths[iid])).unsqueeze(0).to(device)
        heat, prob = cam(x, mi)
        img = denorm(x[0])
        axes[r, 0].imshow(img); axes[r, 0].set_title(f"{iid} (mel real)"); axes[r, 0].axis("off")
        axes[r, 1].imshow(img); axes[r, 1].imshow(heat, cmap="jet", alpha=0.45)
        axes[r, 1].set_title(f"Grad-CAM  p(mel)={prob:.2f}"); axes[r, 1].axis("off")
    plt.tight_layout()
    plt.savefig("/kaggle/working/gradcam_melanomas.png", dpi=150, bbox_inches="tight"); plt.show()
    print("Grad-CAM guardado. El calor debe caer SOBRE la lesion, no en bordes / reglas / pelos.")
    print("Si el modelo mira artefactos -> shortcut learning -> reportarlo como riesgo de fiabilidad.")

## 7. Exportar el mejor modelo + prueba de inferencia (para la API)

In [ ]:
OUT = "/kaggle/working/best_resnet50_ssl.pth"
torch.save({
    "model_state_dict": best_state,
    "classes": classes,
    "arch": "resnet50",
    "img_size": SUP_RES,
    "init": best_name,
    "macro_f1": float(best_f1),
    "temperature": float(T),
    "mel_screening_threshold": float(thr[ok[0]]) if len(ok) else None,
    "norm_mean": MEAN, "norm_std": STD,
}, OUT)
print("Modelo guardado en", OUT)

# Reconstruir desde el checkpoint y probar inferencia
ckpt = torch.load(OUT, map_location=device, weights_only=False)
best_net = resnet50(weights=None)
best_net.fc = nn.Linear(best_net.fc.in_features, len(ckpt["classes"]))
best_net.load_state_dict(ckpt["model_state_dict"]); best_net.to(device).eval()

@torch.no_grad()
def _infer(img, temp):
    """Inferencia sobre un PIL.Image ya cargado. Devuelve dict listo para API/JSON."""
    x = val_tf(img).unsqueeze(0).to(device)
    logits = (best_net(x) + best_net(torch.flip(x, dims=[3]))) / 2   # TTA hflip
    p = F.softmax(logits / temp, dim=1)[0].cpu()
    i = int(p.argmax())
    out = {"pred": ckpt["classes"][i], "conf": float(p[i]),
           "probs": {c: round(float(p[j]), 4) for j, c in enumerate(ckpt["classes"])}}
    # regla de screening: si la prob de melanoma supera el umbral calibrado, alertar aunque no sea el argmax
    thr_mel = ckpt.get("mel_screening_threshold")
    if thr_mel is not None and "mel" in ckpt["classes"]:
        pm = float(p[ckpt["classes"].index("mel")])
        out["mel_prob"] = round(pm, 4)
        out["mel_alert"] = bool(pm >= thr_mel)  # True -> derivar a especialista
    return out

def predict(image_id, temp=None):
    """Inferencia por image_id de HAM (para pruebas rapidas dentro del notebook)."""
    return _infer(load_rgb(img_paths[image_id]), ckpt["temperature"] if temp is None else temp)

def predict_path(path, temp=None):
    """Inferencia sobre CUALQUIER ruta de imagen -> esto es lo que consumira la API."""
    return _infer(load_rgb(path), ckpt["temperature"] if temp is None else temp)

# Prueba: un melanoma real del set
mel_id = meta[meta.dx == "mel"].image_id.iloc[0]
res = predict(mel_id)
print(f"\nPrueba inferencia -> Real: mel | {res}")
print("\nUso en la API:  predict_path('/ruta/a/imagen.jpg')")
print("Descarga", OUT, "(+ RESULTS_SUMMARY.json) desde Output para servir el modelo.")

## 8. RESUMEN CONSOLIDADO DE TODOS LOS RESULTADOS

Junta todo en un solo JSON (`/kaggle/working/RESULTS_SUMMARY.json`) y lo imprime. **Pegá esta salida en el chat** para revisión: dataset, ablación, métricas finales + IC95%, recall por clase, screening de melanoma, abstención, linear probe, validación externa y checklist de KPIs.

In [ ]:
import json
from sklearn.metrics import recall_score

rec = recall_score(val_labels, pred0, average=None, labels=list(range(len(classes))), zero_division=0)

def _m(name):
    mm, _, _ = metrics_from_logits(results[name][2], results[name][3])
    return {"macro_f1": round(float(mm["macro_f1"]), 4), "auc": round(float(mm["auc"]), 4),
            "ece": round(float(mm["ece"]), 4)}

summary = {
    "dataset": {
        "train_imgs": len(tr_ids), "val_imgs": len(va_ids),
        "classes": classes, "lesion_overlap_train_val": int(len(overlap)),
        "split": "StratifiedGroupKFold por lesion_id (anti-leakage)",
    },
    "ssl": {"metodo": "SimSiam", "backbone": "resnet50", "epochs": SSL_EPOCHS,
            "res": SSL_RES, "batch": SSL_BATCH, "run": bool(RUN_SSL)},
    "ablacion_full_finetune": {n: _m(n) for n in results},
    "modelo_entregable": best_name,                 # el que se evalua + exporta = SSL
    "ablacion_ganador_por_f1": ablation_winner,     # solo referencia numerica
    "final_metrics": {
        "macro_f1": round(float(m0["macro_f1"]), 4), "acc": round(float(m0["acc"]), 4),
        "auc": round(float(m0["auc"]), 4),
        "ece_raw": round(float(m0["ece"]), 4), "ece_calibrated": round(float(m_cal["ece"]), 4),
        "temperature": round(float(T), 4),
        "macro_f1_ci95": [round(lo_f1, 4), round(hi_f1, 4)],
        "mel_sens_argmax_ci95": [round(lo_ms, 4), round(hi_ms, 4)],
    },
    "recall_por_clase": {classes[i]: round(float(rec[i]), 4) for i in range(len(classes))},
    "mel_screening": {
        "threshold": round(float(thr[ok[0]]), 4) if len(ok) else None,
        "sensibilidad": round(float(tpr[ok[0]]), 4) if len(ok) else None,
        "especificidad": round(float(1 - fpr[ok[0]]), 4) if len(ok) else None,
    },
    "abstencion": {f"{k:.1f}": {"cobertura": round(v[0], 4), "exactitud": round(v[1], 4)}
                   for k, v in abst.items()},
    "linear_probe": ({a: {f"{int(f*100)}pct": {"macro_f1": round(probe[a][f][0], 4),
                                                "auc": round(probe[a][f][1], 4)} for f in probe[a]}
                      for a in probe} if RUN_SSL else "omitido"),
    "validacion_externa": ext_result if ext_result else "no ejecutada (falta dataset externo etiquetado)",
    "kpis": {
        "macro_f1>=0.65": bool(m0["macro_f1"] >= 0.65),
        "auc>=0.85": bool(m0["auc"] >= 0.85),
        "mel_sens>=0.80": bool((tpr[ok[0]] if len(ok) else 0) >= 0.80),
        "ece_calibrado<=0.10": bool(m_cal["ece"] <= 0.10),
    },
    "artefactos_guardados": [
        "/kaggle/working/best_resnet50_ssl.pth",
        "/kaggle/working/backbone_domain_adaptive_r50.pth",
        "/kaggle/working/reliability_diagram.png",
        "/kaggle/working/gradcam_melanomas.png",
        "/kaggle/working/RESULTS_SUMMARY.json",
    ],
}

with open("/kaggle/working/RESULTS_SUMMARY.json", "w") as f:
    json.dump(summary, f, indent=2, ensure_ascii=False)

print("=" * 60)
print("RESUMEN CONSOLIDADO — pegá esto en el chat para revisión")
print("=" * 60)
print(json.dumps(summary, indent=2, ensure_ascii=False))